# 🧠 08. Treinamento do Modelo - xgboost com dados processados

O objetivo do notebook é simular dados reais com o modelo xgboost

## 🛠️  Imports e Configurações

### Pacotes

In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))


In [6]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from dotenv import load_dotenv
import os
import sys

from src.tools import feature_engineering_completa


### Configurações gerais

In [3]:
sns.set_theme(style="whitegrid")
pd.options.mode.chained_assignment = None

DATA_VENDA = '../data/raw/simular/venda_prevista.csv'         # Input do Cliente
PATH_MODELO = '../models/xgb_model.json'                      # Modelo
DATA_HISTORICO_RAW = '../data/raw/train.csv' 
FILE_SAIDA = '../data/processed/sugestao_compras_final.csv'

CURRENT_DIR = os.getcwd()
sys.path.append(CURRENT_DIR)
PROJECT_ROOT = os.path.dirname(CURRENT_DIR)

dotenv_path = os.path.join(PROJECT_ROOT, '.env')
load_dotenv(dotenv_path)
MARGEM_SEGURANCA = int(os.getenv("MARGEM_SEGURANCA", 10))

# Verificação de segurança antes de rodar
if not os.path.exists(DATA_HISTORICO_RAW):
    print(f"⚠️ AVISO: Não achei '{DATA_HISTORICO_RAW}'.")
    print("Por favor, altere a variável DATA_HISTORICO_RAW para o caminho do seu CSV original de vendas.")


In [4]:
# PIPELINE DE EXECUÇÃO 


print(" Carregamento ---")

# Input Futuro
print(f"Lendo Input: {DATA_VENDA}")
df_input = pd.read_csv(DATA_VENDA)
mapa = {'store':'store', 'loja':'store', 'item':'item', 'ds':'date', 'date':'date'}
df_input.rename(columns=mapa, inplace=True)
df_input['date'] = pd.to_datetime(df_input['date'])
df_input['sales'] = np.nan
df_input['tipo'] = 'futuro'

# Histórico (CSV ORIGINAL)
print(f"Lendo Histórico Original: {DATA_HISTORICO_RAW}")
try:
    df_hist = pd.read_csv(DATA_HISTORICO_RAW)
    
    # Padronização de nomes (caso o csv original use 'date', 'store', 'item')
    # Se o seu csv usar nomes diferentes, ajuste aqui
    if 'date' in df_hist.columns: df_hist['date'] = pd.to_datetime(df_hist['date'])
    
    # Seleção inteligente (Últimos 6 meses apenas)
    cols = ['date', 'store', 'item', 'sales']
    df_hist = df_hist[cols]
    
    corte = df_hist['date'].max() - pd.DateOffset(months=6)
    df_hist = df_hist[df_hist['date'] >= corte]
    df_hist['tipo'] = 'historico'
    print(f"✅ Histórico carregado com sucesso: {df_hist.shape[0]} linhas.")

except Exception as e:
    print(f"❌ ERRO: Não consegui ler o CSV original. Verifique o caminho em DATA_HISTORICO_RAW.")
    print(f"Detalhe do erro: {e}")
    # Cria dummy só para não crashar imediatamente
    df_hist = pd.DataFrame(columns=['date', 'store', 'item', 'sales', 'tipo'])


 Carregamento ---
Lendo Input: ../data/raw/simular/venda_prevista.csv
Lendo Histórico Original: ../data/raw/train.csv
✅ Histórico carregado com sucesso: 92500 linhas.


In [7]:
# Engenharia
print("\n--- ETAPA 2: Engenharia de Features ---")
if not df_hist.empty:
    df_full = pd.concat([df_hist, df_input], ignore_index=True)
    df_full = feature_engineering_completa(df_full, target_col='sales')
    
    # Predição ---
    print("\n--- ETAPA 3: Predição ---")
    df_pred = df_full[df_full['tipo'] == 'futuro'].copy()
    df_pred = df_pred.fillna(0)

    cols_model = [
        'store', 'item', 'month', 'day', 'day_of_week', 'day_of_year',
        'is_weekend', 'is_payday', 'is_holiday', 'days_until_holiday',
        'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos',
        'day_of_year_sin', 'day_of_year_cos', 'lag_1', 'lag_2', 'lag_3',
        'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_91',
        'rolling_mean_7', 'rolling_std_7', 'rolling_mean_28',
        'rolling_std_28', 'rolling_mean_91', 'rolling_std_91',
        'sales_diff_lag1', 'sales_diff_lag7'
    ]


    print("Carregando Modelo...")
    modelo = XGBRegressor()
    modelo.load_model(PATH_MODELO)
    
    print("Calculando...")
    df_pred['y_pred'] = modelo.predict(df_pred[cols_model])
    df_pred['y_pred'] = df_pred['y_pred'].apply(lambda x: x if x > 0 else 0)


    # ETAPA 4: Ordem de Compra (Visualização Melhorada) ---
    print("\n--- ETAPA 4: Gerando Relatório Executivo ---")

    # 1. Definição de Parâmetros
    # MARGEM_SEGURANCA = 12
    np.random.seed(42) # Mantendo a simulação de estoque fixa

    # 2. Tratamento da Previsão (Arredondamento solicitado)
    # Arredondamos para cima (ceil) para não correr risco de falta
    df_pred['Venda_Prevista_Arred'] = np.ceil(df_pred['y_pred']).astype(int)

    # 3. Simulação do Estoque Atual (Em prod, viria do ERP)
    df_pred['Saldo_Atual'] = np.random.randint(0, 20, size=len(df_pred))

    # 4. Cálculo Transparente (Passo a Passo)

    # A. Coluna Explicativa da Margem
    df_pred['Margem_Seguranca'] = MARGEM_SEGURANCA

    # B. Meta de Estoque (Onde queremos chegar)
    # Lógica: "Se vou vender X e quero ter sobra de Y, preciso ter Z na loja"
    df_pred['Meta_Estoque'] = df_pred['Venda_Prevista_Arred'] + df_pred['Margem_Seguranca']

    # C. Cálculo do Pedido Final
    # Lógica: "Preciso de Z, mas já tenho A. Compro a diferença."
    df_pred['Sugestao_Compra'] = df_pred['Meta_Estoque'] - df_pred['Saldo_Atual']

    # Tratamento: Se o resultado for negativo (sobra estoque), o pedido é 0
    df_pred['Sugestao_Compra'] = df_pred['Sugestao_Compra'].clip(lower=0)

    # 5. Formatação Final da Tabela (O que o cliente vai ver)
    # Selecionamos as colunas na ordem lógica de leitura da esquerda para a direita
    colunas_finais = [
        'date', 
        'store', 
        'item', 
        'Venda_Prevista_Arred', 
        'Margem_Seguranca', 
        'Meta_Estoque', 
        'Saldo_Atual', 
        'Sugestao_Compra'
    ]

    df_final = df_pred[colunas_finais].copy()

    # Renomeando para termos de negócio amigáveis
    df_final.rename(columns={
        'date': 'Data',
        'store': 'Loja',
        'item': 'Produto',
        'Venda_Prevista_Arred': 'Previsão_Vendas', # Arredondado
        'Margem_Seguranca': 'Margem_Seg',
        'Meta_Estoque': 'Estoque_Alvo',    # (Previsão + Margem)
        'Saldo_Atual': 'Estoque_Loja',
        'Sugestao_Compra': 'PEDIDO_FINAL'  # O Resultado
    }, inplace=True)

    # (Opcional) Formata data para brasileiro (DD/MM/AAAA)
    df_final['Data'] = df_final['Data'].dt.strftime('%d/%m/%Y')

    print("Relatório de Sugestão de Compras (Detalhado):")
    display(df_final.head(10))

    # Exportação
    df_final.to_csv(FILE_SAIDA, index=False)
    print(f"\nArquivo gerado com sucesso: {FILE_SAIDA}")

else:
    print("Processo interrompido por falta de histórico.")


--- ETAPA 2: Engenharia de Features ---

--- ETAPA 3: Predição ---
Carregando Modelo...
Calculando...

--- ETAPA 4: Gerando Relatório Executivo ---
Relatório de Sugestão de Compras (Detalhado):


,Data,Loja,Produto,Previsão_Vendas,Margem_Seg,Estoque_Alvo,Estoque_Loja,PEDIDO_FINAL
92500,01/01/2018,1,1,5,11,16,6,10
92504,05/01/2018,1,1,2,11,13,19,0
92507,08/01/2018,1,1,2,11,13,14,0
92502,03/01/2018,1,2,5,11,16,10,6
92501,02/01/2018,1,4,5,11,16,7,9
92506,07/01/2018,1,4,2,11,13,6,7
92503,04/01/2018,1,6,5,11,16,18,0
92505,06/01/2018,1,9,5,11,16,10,6



Arquivo gerado com sucesso: ../data/processed/sugestao_compras_final.csv
